<a href="https://colab.research.google.com/github/JordanDCunha/Introduction-to-Machine-Learning-with-Python/blob/main/Chapter5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 5: Model Evaluation and Improvement

## Overview
- This chapter focuses on **evaluating machine learning models** and **improving their performance**
  by selecting appropriate parameters.
- The emphasis is on **supervised learning**:
  - Regression
  - Classification
- Evaluation in **unsupervised learning** is often qualitative and exploratory, so it is not the main
  focus here.

## Train–Test Split Recap
- So far, supervised models have been evaluated using:
  - A **training set** to fit the model
  - A **test set** to measure performance on unseen data
- The typical workflow:
  - Split data using `train_test_split`
  - Train the model with `fit`
  - Evaluate with `score`
- For classification:
  - `score` computes **accuracy** (fraction of correctly classified samples)

## Why Evaluation Matters
- The goal is **generalization**, not memorizing the training data
- We care about how well the model performs on **new, unseen data**
- High training accuracy alone is not meaningful

## What This Chapter Adds
- **Cross-validation**
  - A more robust method for estimating generalization performance
- **Advanced ev**


In [ ]:
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Create a synthetic dataset
X, y = make_blobs(random_state=0)

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0
)

# Train a logistic regression model
logreg = LogisticRegression()
logreg.fit(X_train, y_train)

# Evaluate the model on the test set
print("Test set score: {:.2f}".format(logreg.score(X_test, y_test)))


# 5.1 Cross-Validation

Cross-validation is a statistical method for evaluating how well a machine learning model
generalizes to unseen data. It is more stable and reliable than using a single
train–test split.

Instead of splitting the dataset once, cross-validation repeatedly splits the data
into training and test sets and trains multiple models.

The most common approach is **k-fold cross-validation**, where:
- The dataset is split into *k* equally sized folds (typically 5 or 10)
- Each fold is used once as the test set
- The remaining folds are used as the training set
- Performance is averaged across all folds


In [ ]:
import mglearn
mglearn.plots.plot_cross_validation()


import mglearn
mglearn.plots.plot_cross_validation()


In five-fold cross-validation:
- The dataset is split into five folds
- Five models are trained
- Each model is evaluated on a different fold
- Every sample appears in the test set exactly once

This produces multiple accuracy values instead of just one.


## 5.1.1 Cross-Validation in scikit-learn

In scikit-learn, cross-validation is implemented using
`cross_val_score` from `sklearn.model_selection`.

It requires:
- A model
- Feature data
- Target labels


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

iris = load_iris()
logreg = LogisticRegression()

scores = cross_val_score(logreg, iris.data, iris.target)
print("Cross-validation scores:", scores)


By default:
- Older versions of scikit-learn use **3-fold** cross-validation
- Newer versions (≥ 0.22) use **5-fold**

You can control the number of folds using the `cv` parameter.


In [ ]:
scores = cross_val_score(logreg, iris.data, iris.target, cv=5)
print("Cross-validation scores:", scores)


A common way to summarize cross-validation results is by computing the mean score.


In [ ]:
print("Average cross-validation score: {:.2f}".format(scores.mean()))


The variation between fold scores gives insight into:
- Dataset size
- Model stability
- Sensitivity to training data selection


### Using `cross_validate`

`cross_validate` provides additional information:
- Training time
- Scoring time
- Training score (optional)
- Test score


In [ ]:
from sklearn.model_selection import cross_validate

res = cross_validate(
    logreg, iris.data, iris.target,
    cv=5, return_train_score=True
)
res


In [ ]:
import pandas as pd

res_df = pd.DataFrame(res)
display(res_df)
print("Mean values:\n", res_df.mean())


## 5.1.2 Benefits of Cross-Validation

Advantages:
- Reduces dependency on a single lucky/unlucky split
- Every sample is tested exactly once
- Provides insight into best-case and worst-case performance
- Uses data more efficiently

Disadvantage:
- Computationally more expensive (k models instead of one)

Important:
Cross-validation does **not** return a trained model.
It is only used for evaluation.


## 5.1.3 Stratified k-Fold Cross-Validation

In classification tasks, datasets are often ordered by class label.
Using standard k-fold splitting can lead to folds containing only one class.

This causes misleading or useless evaluation results.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
print("Iris labels:\n", iris.target)


To solve this, scikit-learn uses **stratified k-fold cross-validation** for classification.

Stratification ensures:
- Each fold has the same class proportions as the full dataset


In [ ]:
mglearn.plots.plot_stratified_cross_validation()


For regression tasks, standard k-fold cross-validation is used by default.


### Explicit Cross-Validation Strategies

You can manually specify a splitting strategy using splitter classes like `KFold`.


In [ ]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5)
print(cross_val_score(logreg, iris.data, iris.target, cv=kfold))


Using non-stratified k-fold on classification data can be disastrous.


In [ ]:
kfold = KFold(n_splits=3)
print(cross_val_score(logreg, iris.data, iris.target, cv=kfold))


Shuffling the data can mitigate ordering issues.


In [ ]:
kfold = KFold(n_splits=3, shuffle=True, random_state=0)
print(cross_val_score(logreg, iris.data, iris.target, cv=kfold))


### Leave-One-Out Cross-Validation (LOO)

- Each fold contains a single test sample
- Very expensive for large datasets
- Useful for small datasets


In [ ]:
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()
scores = cross_val_score(logreg, iris.data, iris.target, cv=loo)

print("Number of iterations:", len(scores))
print("Mean accuracy:", scores.mean())


### Shuffle-Split Cross-Validation

- Randomly samples training and test sets
- Repeats splitting multiple times
- Useful for large datasets or subsampling


In [ ]:
from sklearn.model_selection import ShuffleSplit

shuffle_split = ShuffleSplit(
    train_size=0.5,
    test_size=0.5,
    n_splits=10
)

scores = cross_val_score(logreg, iris.data, iris.target, cv=shuffle_split)
print(scores)


A stratified variant exists for classification:
`StratifiedShuffleSplit`


### Cross-Validation with Groups

Some datasets contain related samples (e.g., multiple samples from the same person).

To prevent data leakage:
- Entire groups must stay together
- Use `GroupKFold`


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=12, random_state=0)
groups = [0, 0, 0, 1, 1, 1, 1, 2, 2, 3, 3, 3]

scores = cross_val_score(
    logreg, X, y,
    groups=groups,
    cv=GroupKFold(n_splits=3)
)

print(scores)


In [ ]:
mglearn.plots.plot_group_kfold()


### Summary of Cross-Validation Strategies

Most commonly used:
- KFold (regression)
- StratifiedKFold (classification)
- GroupKFold (grouped data)

Cross-validation is a core tool for reliable model evaluation.


# 5.2 Grid Search

Grid search is a systematic way to find the best hyperparameter values
for a machine learning model.

Key ideas:
- Most models have parameters that strongly affect performance
- Finding good parameter values is essential for generalization
- Grid search tries *all combinations* of specified parameter values


## Example: Kernel SVM with RBF Kernel

Important parameters for RBF SVM:
- **C**: regularization parameter
- **gamma**: kernel bandwidth

In this example:
- C ∈ {0.001, 0.01, 0.1, 1, 10, 100}
- gamma ∈ {0.001, 0.01, 0.1, 1, 10, 100}
- Total combinations = 6 × 6 = 36


## 5.2.1 Simple Grid Search

A naive grid search:
- Uses nested loops over parameters
- Trains a model for each combination
- Evaluates each model on a test set
- Keeps the best-performing parameters


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, random_state=0)

print("Size of training set:", X_train.shape[0])
print("Size of test set:", X_test.shape[0])

best_score = 0

for gamma in [0.001, 0.01, 0.1, 1, 10, 100]:
    for C in [0.001, 0.01, 0.1, 1, 10, 100]:
        svm = SVC(gamma=gamma, C=C)
        svm.fit(X_train, y_train)
        score = svm.score(X_test, y_test)

        if score > best_score:
            best_score = score
            best_parameters = {'C': C, 'gamma': gamma}

print("Best score:", best_score)
print("Best parameters:", best_parameters)


## 5.2.2 Danger of Overfitting the Parameters

Problem:
- Test set was used to choose parameters
- This causes optimistic (incorrect) performance estimates

Solution:
- Split data into **three sets**:
  - Training set → fit model
  - Validation set → choose parameters
  - Test set → final evaluation only


In [ ]:
# split data into train+validation and test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    iris.data, iris.target, random_state=0)

# split train+validation into training and validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, random_state=1)

print("Training set:", X_train.shape[0])
print("Validation set:", X_valid.shape[0])
print("Test set:", X_test.shape[0])


In [ ]:
best_score = 0

for gamma in [0.001, 0.01, 0.1, 1, 10, 100]:
    for C in [0.001, 0.01, 0.1, 1, 10, 100]:
        svm = SVC(gamma=gamma, C=C)
        svm.fit(X_train, y_train)
        score = svm.score(X_valid, y_valid)

        if score > best_score:
            best_score = score
            best_parameters = {'C': C, 'gamma': gamma}

svm = SVC(**best_parameters)
svm.fit(X_trainval, y_trainval)
test_score = svm.score(X_test, y_test)

print("Best validation score:", best_score)
print("Best parameters:", best_parameters)
print("Test set score:", test_score)


Key takeaway:
- Validation accuracy ≠ test accuracy
- Test set must be used **once only**
- Any parameter tuning using the test set causes data leakage


## 5.2.3 Grid Search with Cross-Validation

Instead of a single validation split:
- Use cross-validation for parameter evaluation
- Produces more stable estimates
- Reduces sensitivity to data splits

Downside:
- Computationally expensive


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score

best_score = 0

for gamma in [0.001, 0.01, 0.1, 1, 10, 100]:
    for C in [0.001, 0.01, 0.1, 1, 10, 100]:
        svm = SVC(gamma=gamma, C=C)
        scores = cross_val_score(svm, X_trainval, y_trainval, cv=5)
        score = np.mean(scores)

        if score > best_score:
            best_score = score
            best_parameters = {'C': C, 'gamma': gamma}

svm = SVC(**best_parameters)
svm.fit(X_trainval, y_trainval)


Grid search with cross-validation:
- Trains k models per parameter combination
- Example: 36 combinations × 5 folds = 180 models


## GridSearchCV

GridSearchCV:
- Automates grid search with cross-validation
- Acts like a regular estimator
- Re-trains model using best parameters


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(SVC(), param_grid, cv=5, return_train_score=True)

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, random_state=0)

grid_search.fit(X_train, y_train)


In [ ]:
print("Test set score:", grid_search.score(X_test, y_test))
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)


Important attributes:
- best_params_ → best hyperparameters
- best_score_ → mean cross-validation score
- best_estimator_ → trained model with best parameters


In [ ]:
print(grid_search.best_estimator_)


## Analyzing Grid Search Results

The full results are stored in:
- cv_results_

Contains:
- Mean test scores
- Per-fold scores
- Standard deviations
- Parameter combinations


In [ ]:
import pandas as pd

results = pd.DataFrame(grid_search.cv_results_)
display(results.head())


Grid search results can be visualized as heat maps
to understand parameter sensitivity and ranges.


## Conditional Parameter Grids

Some parameters depend on others.

Example:
- kernel='rbf' → use C and gamma
- kernel='linear' → only C

Solution:
- Use a list of parameter grids


In [ ]:
param_grid = [
    {'kernel': ['rbf'],
     'C': [0.001, 0.01, 0.1, 1, 10, 100],
     'gamma': [0.001, 0.01, 0.1, 1, 10, 100]},
    {'kernel': ['linear'],
     'C': [0.001, 0.01, 0.1, 1, 10, 100]}
]

grid_search = GridSearchCV(SVC(), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)


## Nested Cross-Validation

Nested cross-validation:
- Outer loop → model evaluation
- Inner loop → parameter tuning
- Produces multiple test scores
- Does NOT return a final model

Use case:
- Model comparison
- Reliable performance estimation


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    GridSearchCV(SVC(), param_grid, cv=5),
    iris.data, iris.target, cv=5
)

print("Nested CV scores:", scores)
print("Mean score:", scores.mean())


## Parallelization

Grid search and cross-validation are embarrassingly parallel.

Options:
- n_jobs = -1 → use all CPU cores
- Monitor memory usage for large jobs
- Distributed computing possible via Dask


# 5.3 Evaluation Metrics and Scoring

- Accuracy (classification) and R² (regression) are common but often insufficient
- Real-world ML decisions depend on *business goals*, not just predictive accuracy
- Choosing the right evaluation metric is critical for meaningful model comparison
- Metrics must reflect the *cost of errors* and *class imbalance*


## 5.3.1 Keep the End Goal in Mind

- ML predictions are usually part of a larger decision-making system
- The *business metric* defines success (e.g., fewer accidents, higher revenue)
- Model choice affects *business impact*
- Directly measuring business impact is often impractical during development
- Surrogate metrics (accuracy, recall, AUC, etc.) are used instead
- Choose the metric that best approximates the real-world objective


## 5.3.2 Metrics for Binary Classification

### Types of Errors
- **False Positive (FP)**: predicting positive when the truth is negative
- **False Negative (FN)**: predicting negative when the truth is positive
- Consequences of FP and FN are often very different
- Example: cancer screening prioritizes minimizing false negatives


### Imbalanced Datasets

- One class appears much more frequently than the other
- Accuracy becomes misleading in this setting
- Example: predicting "no click" always can give very high accuracy
- Alternative metrics are required to detect useful models


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

digits = load_digits()
y = digits.target == 9  # binary classification: nine vs rest

X_train, X_test, y_train, y_test = train_test_split(
    digits.data, y, random_state=0
)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print("Accuracy:", dummy.score(X_test, y_test))


## Confusion Matrix

- A 2×2 table summarizing prediction outcomes
- Rows = true labels
- Columns = predicted labels
- Diagonal = correct predictions
- Off-diagonal = errors
- Provides more insight than accuracy alone


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

pred = logreg.predict(X_test)
print(confusion_matrix(y_test, pred))


## Precision, Recall, and F1-Score

- **Precision** = TP / (TP + FP)
  - Measures correctness of positive predictions
- **Recall** = TP / (TP + FN)
  - Measures how many positives were found
- Precision–Recall tradeoff always exists
- **F1-score** is the harmonic mean of precision and recall
- F1 is more informative than accuracy for imbalanced data


In [ ]:
from sklearn.metrics import f1_score, classification_report

print("F1 score:", f1_score(y_test, pred))
print(classification_report(y_test, pred, target_names=["not nine", "nine"]))


## Decision Thresholds and Uncertainty

- Classifiers often output scores or probabilities
- Default thresholds:
  - decision_function → 0
  - predict_proba → 0.5
- Changing thresholds adjusts precision–recall tradeoff
- Thresholds must be tuned using validation data, not test data


In [ ]:
from sklearn.svm import SVC

svc = SVC(gamma=0.05)
svc.fit(X_train, y_train)

y_pred_default = svc.predict(X_test)
y_pred_custom = svc.decision_function(X_test) > -0.8

print("Default threshold:")
print(classification_report(y_test, y_pred_default))

print("Lower threshold:")
print(classification_report(y_test, y_pred_custom))


## Precision-Recall and ROC Curves

- Curves evaluate performance across all thresholds
- Precision-Recall curve:
  - Best for imbalanced datasets
  - Summary metric: Average Precision
- ROC curve:
  - Plots TPR vs FPR
  - Summary metric: AUC
- Random guessing:
  - PR baseline = fraction of positives
  - ROC AUC = 0.5


In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, svc.decision_function(X_test))
print("ROC AUC:", auc)


## Multiclass Classification Metrics

- Metrics are extensions of binary metrics
- Confusion matrix and classification report still apply
- Multiclass F1 averaging strategies:
  - micro: per-sample importance
  - macro: per-class importance
  - weighted: class-size weighted


In [ ]:
from sklearn.metrics import f1_score

y_multi = digits.target
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, y_multi, random_state=0
)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
pred = lr.predict(X_test)

print("Micro F1:", f1_score(y_test, pred, average="micro"))
print("Macro F1:", f1_score(y_test, pred, average="macro"))


## Using Evaluation Metrics in Model Selection

- Metrics can be specified via the `scoring` parameter
- Used in:
  - cross_val_score
  - GridSearchCV
- Metric choice affects which model is selected
- Always align scoring metric with business goal


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {"gamma": [0.001, 0.01, 0.1, 1]}
grid = GridSearchCV(SVC(), param_grid, scoring="average_precision")
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)


# 5.4 Summary and Outlook

- Cross-validation, grid search, and evaluation metrics are core tools for improving ML models
- These techniques, combined with supervised and unsupervised algorithms, form the foundation of practical machine learning
- Model evaluation aims to estimate how a model will perform on *future, unseen data*

## Key Point 1: Proper Use of Cross-Validation and Test Data

- Test sets and cross-validation estimate future generalization performance
- Using test data to *select* models or parameters leads to overly optimistic results
- Once test data influences decisions, it is no longer a true test set
- Correct data usage requires:
  - **Training data**: fit models
  - **Validation data**: select models and parameters
  - **Test data**: final evaluation only
- Cross-validation can replace any of these splits
- Common best practice:
  - Use cross-validation on the training set for model selection
  - Use a held-out test set only once for final evaluation

## Key Point 2: Importance of the Evaluation Metric

- High accuracy is rarely the true goal of a machine learning task
- Metrics should reflect the *real-world consequences* of predictions
- Classification problems are often imbalanced
- False positives and false negatives usually have different costs
- Choosing the wrong metric can result in selecting a harmful or useless model
- Always align the evaluation metric with how the model will actually be used

## Looking Ahead

- Grid search and cross-validation so far apply to single supervised models
- Many real-world models require preprocessing and feature transformations
- Different data representations can significantly improve performance
- The next chapter introduces **Pipelines**
  - Allow chaining preprocessing and modeling steps
  - Enable grid search and cross-validation over entire workflows
